**Eye-Tracking & Neurocognitive Intelligence | Eye-Tracking Foundations & Signal Analysis | Presented by: Reza Saadatyar (2026)**

**02 | Gap-Aware Gaze Preprocessing and Velocity**

![Conceptual workflow for stage 02](../../images/02_gap_aware_gaze_preprocessing.svg)

**Objective:** Produce a transparent valid-sample representation without bridging gaps and calculate velocity in dva/s.

**Dataset:** Only the named current CSV saved by Notebook 01.

**Notebook scope:** The smooth trace is optional and retains the raw x/y samples; no artificial blinks are reconstructed.

**Execution status:** Source code prepared. No original-data results are claimed until this notebook is executed with the specified release.


**00. Colab Repository Setup**

In [ ]:
# ======================== 00. Colab Repository Setup =========================

# -------------------------- 1. Repository Checkout ---------------------------
from pathlib import Path
import subprocess

repo_url = "https://github.com/RezaSaadatyar/Eye-Tracking-Neurocognitive-Intelligence.git"
repo_path = Path("/content") / repo_url.rsplit("/", 1)[-1].removesuffix(".git")
if not repo_path.exists():
    subprocess.run(["git", "clone", "-q", repo_url, str(repo_path)], check=True)

# ---------------------------- 2. Open Repository -----------------------------
%cd {repo_path}


**01. Imports and Research Environment**

In [ ]:
# =================== 01. Imports and Research Environment ====================

# ------------------------ 1. Study Environment Setup -------------------------
from python_modules.core.colab_setup import setup_colab
root = setup_colab("eye_tracking_foundations")

# ---------------------------- 2. Standard Library ----------------------------
from pathlib import Path

# ----------------- 3. Scientific Computing and Data Analysis -----------------
import numpy as np
import pandas as pd

# ------------------- 4. Visualisation and Notebook Display -------------------
import matplotlib.pyplot as plt
from IPython.display import display
from core.plot_style import plot_style, save_figure
from core.output_download import download_outputs_from_colab

# ------------------------ 5. Study Research Utilities ------------------------
from core.stage_outputs import ensure_stage_inputs
from gaze_source import read_stage
from gaze_preprocessing import preprocess_samples

# --------------------- 6. Output and Presentation Setup ----------------------
output_dir = root / "eye_tracking_foundations" / "output"
output_dir.mkdir(parents=True, exist_ok=True)
plot_style(
    font_family="DejaVu Sans", title_size=12.5,
    title_weight="normal", label_size=11.5,
    label_weight="normal", tick_size=11,
    legend_size=10, figure_dpi=100, figure_size=(10, 4),
)
print("Current outputs:", output_dir)


**02. Recover Notebook 01 Current Inputs**

Reuse ~01_gaze_samples.csv~. If using another Colab runtime, upload the ZIP
produced by Notebook 01 to the exact path specified by the stage helper.
No original data extraction is needed at this stage.


In [ ]:
# ================== 02. Recover Notebook 01 Current Inputs ===================

ensure_stage_inputs(output_dir, 1, ("01_gaze_samples.csv",))
samples = read_stage(
    output_dir / "01_gaze_samples.csv",
    ("record_id", "sample_index", "n", "x", "y", "val"),
)
display(samples[["record_id", "n", "x", "y", "val"]].head())


**03. Apply Gap-Respecting Smoothing**

For the first method demonstration, use a five-sample centred median within
eligible uninterrupted segments, with a conservative 5-ms gap boundary.
These are explicitly labelled *initial analysis settings*, not tuned
thresholds or evidence that the instrument has a universal noise profile.
Retain raw gaze, source ~lab~ and sample identifiers.


In [ ]:
# ==================== 03. Apply Gap-Respecting Smoothing =====================

SMOOTHING_SAMPLES = 5
MAX_GAP_MS = 5.0
prepared = preprocess_samples(
    samples, smoothing_samples=SMOOTHING_SAMPLES,
    max_gap_ms=MAX_GAP_MS,
)
assert not prepared.duplicated(["record_id", "sample_index"]).any()
prepared.to_csv(output_dir / "02_preprocessed_samples.csv", index=False)
print("Prepared samples:", len(prepared))


**04. Inspect Segments and Velocity**

Velocity is angular distance per second between consecutive smoothed
valid samples within each segment. It is undefined at segment starts,
not silently filled with zero. A gap or invalid sample breaks continuity.


In [ ]:
# ===================== 04. Inspect Segments and Velocity =====================

segments = prepared.groupby("record_id", as_index=False).agg(
    n_samples=("sample_index", "size"),
    n_valid=("valid_for_motion", "sum"),
    n_segments=("segment_id", "nunique"),
    median_velocity_dva_s=("velocity_dva_s", "median"),
    max_velocity_dva_s=("velocity_dva_s", "max"),
)
segments.to_csv(output_dir / "02_preprocessing_summary.csv", index=False)
display(segments)


**05. Compare Raw and Processed Gaze**

Show a bounded interval only. Smoothing is introduced for methodological
comparison, not because it is known to improve event boundaries. Its effect
will be assessed in the later sensitivity notebook.


In [ ]:
# ==================== 05. Compare Raw and Processed Gaze =====================

record_id = str(prepared.record_id.iloc[0])
part = prepared[prepared.record_id.eq(record_id)].head(3000)
fig, axes = plt.subplots(2,1, figsize=(11,5), sharex=True)
axes[0].plot(part.time_s, part.x, lw=0.6, label="raw x")
axes[0].plot(part.time_s, part.x_smooth, lw=0.8, label="median-smoothed x")
axes[0].set_ylabel("Horizontal gaze (dva)")
axes[0].legend()
axes[1].plot(part.time_s, part.velocity_dva_s, lw=0.7)
axes[1].set_ylabel("Velocity (dva/s)")
axes[1].set_xlabel("Time from record start (s)")
fig.suptitle(f"Gap-aware motion representation: {record_id}")
fig.tight_layout()
save_figure(fig, output_dir / "02_velocity_profile.png")
plt.show()


**06. Export Reusable Numerical Inputs**

Notebook 03 reads ~02_preprocessed_samples.csv~ directly. The stage ZIP
contains exactly the named Notebook 02 outputs for reuse in later runtimes.


In [ ]:
# =================== 06. Export Reusable Numerical Inputs ====================

print("Missing velocities at segment starts remain NaN.")
print("Current motion representation:", output_dir / "02_preprocessed_samples.csv")
download_outputs_from_colab("eye_tracking_foundations/output", "02")
